# 33 — Quantile Regression & Prediction Intervals (AUB + Peers)

Trains **three LightGBM quantile regressors** (q10 / q50 / q90) on the **full
merged dataset (AUB + peer institutions)** using the same feature pipeline as
notebook 32.  The resulting prediction interval is used to make a
**three-way decision**:

| Interval vs threshold (log-space) | Decision |
|---|---|
| upper bound < log1p(thr_75) | Definitely Low Impact |
| lower bound > log1p(thr_75) | Definitely High Impact |
| interval straddles threshold | Uncertain |

**Goals:**
1. Calibrate the 80 % interval (q10–q90) and measure actual coverage
2. Compare definite-prediction F1 vs point-prediction baseline
3. Save q50 model as a drop-in replacement for the existing `lightgbm.pkl`

In [ ]:
import sys
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')

PROJECT_ROOT   = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH    = PROJECT_ROOT / 'data' / 'processed' / 'all_unis_cleaned.pkl'
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features'
MODELS_DIR   = PROJECT_ROOT / 'models' / 'regression'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE   = 42
TRAIN_YEARS    = list(range(2010, 2018))  # expand train; keep test on citation-mature papers
TEST_YEARS     = list(range(2018, 2021))  # 2018-2020: 6-8 years to accumulate citations
TFIDF_MAX_FEAT = 5000

# 50 % CI (q25/q75): narrower interval so high-impact papers can break through
# the threshold on the lower bound and receive a definite "High" call.
# An 80 % CI (q10/q90) has mean width ~2.4 log-units; the p75 threshold sits
# at log1p(43)≈3.78, making lower_log > threshold impossible in practice.
QUANTILES    = [0.25, 0.50, 0.75]
Q_LOW, Q_MID, Q_HIGH = QUANTILES   # 50 % CI bounds + median

NOMINAL_COVERAGE = 0.50   # matches Q_HIGH - Q_LOW

print('Paths OK')

## 1. Load & split data  *(identical to nb32)*

In [ ]:
df = pd.read_pickle(DATA_PATH)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')

# Use ALL institutions (AUB + peers) — no institution filter
df_train = df[df['Year'].isin(TRAIN_YEARS)].dropna(subset=['Abstract', 'Citations']).copy()
df_test  = df[df['Year'].isin(TEST_YEARS)].dropna(subset=['Abstract', 'Citations']).copy()

print(f'Train: {len(df_train):,} papers  |  Test: {len(df_test):,} papers')
print(f'Train citation stats — median: {df_train["Citations"].median():.0f}, '
      f'mean: {df_train["Citations"].mean():.1f}, '
      f'p75: {df_train["Citations"].quantile(0.75):.0f}')
if 'Institution' in df.columns:
    print(f'Institutions in train: {df_train["Institution"].nunique()}')

## 2. Fit TF-IDF & venue stats  *(reuse artifacts from nb32 if present)*

In [ ]:
from src.features.venue_features import compute_venue_statistics

tfidf_path = FEATURES_DIR / 'tfidf_vectorizer.pkl'
venue_path = FEATURES_DIR / 'venue_statistics.pkl'

# Always refit — training set changed, old artifacts are stale
tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEAT, ngram_range=(1, 2),
    min_df=5, max_df=0.90, stop_words='english', sublinear_tf=True
)
tfidf.fit(df_train['Abstract'].fillna(''))
with open(tfidf_path, 'wb') as f:
    pickle.dump(tfidf, f)

venue_stats = compute_venue_statistics(
    df_train, venue_col='Scopus Source title', citation_col='Citations'
)
with open(venue_path, 'wb') as f:
    pickle.dump(venue_stats, f)

print('TF-IDF and venue stats fitted and saved.')

## 3. Build features via deployment pipeline  *(identical to nb32)*

In [ ]:
from src.deployment.prediction_service import CitationPredictor

predictor = CitationPredictor(
    models_dir=str(PROJECT_ROOT / 'models'),
    features_dir=str(FEATURES_DIR)
)
predictor.initialize()
print('Predictor initialised.')

In [ ]:
COL_MAP = {
    'Source title': 'Scopus Source title',
    'H-index':      'Authors H-index',
    'Publication type': 'Publication Type',
    'Source type':  'Source type',
    'Open Access':  'is_open_access',
    'Topic Prominence Percentile': 'topic_prominence',
}

def prep_df(df_in):
    d = df_in.copy().reset_index(drop=True)
    for old, new in COL_MAP.items():
        if old in d.columns and new not in d.columns:
            d.rename(columns={old: new}, inplace=True)
    if 'is_open_access' in d.columns:
        d['is_open_access'] = pd.to_numeric(
            d['is_open_access'].map(
                lambda x: 1 if str(x).strip().lower() not in
                    ('', 'nan', 'none', 'no', '0', 'false') else 0
            ), errors='coerce'
        ).fillna(0).astype(int)
    if 'topic_prominence' not in d.columns and 'Topic Prominence Percentile' in df_in.columns:
        d['topic_prominence'] = pd.to_numeric(
            df_in['Topic Prominence Percentile'].values, errors='coerce'
        ).fillna(50.0)
    return d

df_train_p = prep_df(df_train)
df_test_p  = prep_df(df_test)

citations_train = df_train['Citations'].reset_index(drop=True)
citations_test  = df_test['Citations'].reset_index(drop=True)

X_train_raw = predictor.prepare_features(df_train_p)
X_test_raw  = predictor.prepare_features(df_test_p)

# Drop leaky columns (same as nb32)
YEAR_TOKEN_COLS = [c for c in X_train_raw.columns if re.match(r'tfidf_\d{4}$', c)]
LEAKY = ['venue_avg_citations', 'venue_prestige_score', 'is_top_venue'] + YEAR_TOKEN_COLS
dropped = [c for c in LEAKY if c in X_train_raw.columns]
X_train = X_train_raw.drop(columns=dropped)
X_test  = X_test_raw.drop(columns=dropped)

y_train_log = np.log1p(citations_train.loc[X_train.index].values)
y_test_log  = np.log1p(citations_test.loc[X_test.index].values)

y_train_raw = citations_train.loc[X_train.index].values.astype(float)
y_test_raw  = citations_test.loc[X_test.index].values.astype(float)

thr_75 = float(np.quantile(y_train_raw, 0.75))
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'p75 citation threshold (train): {thr_75:.0f}')

## 4. Train quantile regressors

We fit **three LightGBM quantile models** on log1p citations:

| Model | Quantile | Role |
|---|---|---|
| q25 | 0.25 | Lower bound of the 50 % CI |
| q50 | 0.50 | Point prediction (median) |
| q75 | 0.75 | Upper bound of the 50 % CI |

Using a 50 % CI (q25/q75) keeps interval widths narrow enough that the most
confidently high-citation papers can break through the threshold on the lower bound.

In [ ]:
LGBM_PARAMS = dict(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

import lightgbm as lgb

quantile_models = {}
for q in QUANTILES:
    print(f'Training q={q:.2f} ...', end=' ')
    m = LGBMRegressor(
        objective='quantile',
        alpha=q,
        **LGBM_PARAMS
    )
    m.fit(
        X_train, y_train_log,
        eval_set=[(X_test, y_test_log)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(200)
        ]
    )
    quantile_models[q] = m
    print(f'done (best iter={m.best_iteration_})')

## 5. Calibrate interval coverage

In [ ]:
# Keep predictions in log-space for threshold comparison — exponentiate only for display
pred_log = {}
for q, m in quantile_models.items():
    p = m.predict(X_test)
    pred_log[q] = np.clip(p, 0, None)

# Enforce monotonicity in log-space: q_low <= q50 <= q_high
pred_log[Q_MID]  = np.maximum(pred_log[Q_MID], pred_log[Q_LOW])
pred_log[Q_HIGH] = np.maximum(pred_log[Q_HIGH], pred_log[Q_MID])

lower_log  = pred_log[Q_LOW]
median_log = pred_log[Q_MID]
upper_log  = pred_log[Q_HIGH]

# Raw-scale equivalents for visualisation only
lower  = np.expm1(lower_log)
median = np.expm1(median_log)
upper  = np.expm1(upper_log)

# Empirical coverage — compare in log-space
thr_75_log = np.log1p(thr_75)
covered = ((y_test_log >= lower_log) & (y_test_log <= upper_log))
coverage = covered.mean()
interval_width_log = (upper_log - lower_log).mean()

print(f'Nominal coverage  : {NOMINAL_COVERAGE*100:.0f} %  (50 % CI: q{int(Q_LOW*100)}/q{int(Q_HIGH*100)})')
print(f'Empirical coverage: {coverage*100:.1f} %  (log-space)')
print(f'Mean interval width: {interval_width_log:.3f} log-citations')
print(f'p75 threshold: {thr_75:.0f} citations  →  log1p = {thr_75_log:.3f}')
print()
rho = spearmanr(y_test_log, median_log).statistic
print(f'q50 Spearman vs log citations: {rho:.4f}')

## 6. Three-way classification

The interval comparison is done in **log-space** (the space the models were trained in).

We use a **50 % CI** (q25–q75) rather than 80 % (q10–q90) because:
- The mean 80 % interval width (~2.4 log-units) is wider than the log-space distance
  from the typical prediction to the threshold (log1p≈3.78), so q10 never exceeds the
  threshold and no paper is ever called "High (definite)".
- The 50 % CI is narrower, so confidently high-citation papers can have their
  lower bound exceed the threshold.

| Condition (log-space) | Label |
|---|---|
| `upper_log < log1p(thr_75)` | **Low** (high confidence) |
| `lower_log > log1p(thr_75)` | **High** (high confidence) |
| interval straddles threshold | **Uncertain** |

In [ ]:
true_label = (y_test_log >= thr_75_log).astype(int)   # 1=High, 0=Low (log-space)

# Three-way decision — entirely in log-space
def three_way_log(lower_log, upper_log, thr_log):
    return np.where(
        upper_log < thr_log, 'Low',
        np.where(lower_log > thr_log, 'High', 'Uncertain')
    )

decisions = three_way_log(lower_log, upper_log, thr_75_log)

n_total  = len(decisions)
n_high   = (decisions == 'High').sum()
n_low    = (decisions == 'Low').sum()
n_uncert = (decisions == 'Uncertain').sum()

print('=== Three-way Decision Distribution ===')
print(f'  High      : {n_high:4d}  ({n_high/n_total*100:.1f} %)')
print(f'  Low       : {n_low:4d}  ({n_low/n_total*100:.1f} %)')
print(f'  Uncertain : {n_uncert:4d}  ({n_uncert/n_total*100:.1f} %)')
print(f'  Coverage  : {(n_high+n_low)/n_total*100:.1f} % definite predictions')
print()

definite_mask = decisions != 'Uncertain'
if definite_mask.sum() > 0:
    pred_bin = (decisions == 'High').astype(int)
    d_true = true_label[definite_mask]
    d_pred = pred_bin[definite_mask]
    f1_definite  = f1_score(d_true, d_pred, average='binary', zero_division=0)
    acc_definite = (d_true == d_pred).mean()
    print(f'=== Definite-prediction accuracy (n={definite_mask.sum()}) ===')
    print(f'  F1 (High Impact): {f1_definite:.4f}')
    print(f'  Accuracy        : {acc_definite:.4f}')
    print()
    print(classification_report(d_true, d_pred, target_names=['Low Impact', 'High Impact']))

## 7. Baseline comparison (point prediction from q50)

In [ ]:
# Point-prediction binary classification using q50 (median) in log-space
pred_label_point = (median_log >= thr_75_log).astype(int)
f1_point = f1_score(true_label, pred_label_point, average='binary', zero_division=0)

print('=== Full test set — point prediction (q50 threshold, log-space) ===')
print(classification_report(true_label, pred_label_point,
                             target_names=['Low Impact', 'High Impact']))

print(f'Summary:')
print(f'  Point-prediction F1 (all {n_total} papers)        : {f1_point:.4f}')
if definite_mask.sum() > 0:
    print(f'  Three-way F1     ({definite_mask.sum()} definite papers): {f1_definite:.4f}')
    print(f'  Coverage         : {definite_mask.mean()*100:.1f} %')

## 8. Precision–recall tradeoff: coverage vs F1

In [ ]:
# Sweep gap in log-space around the log-space threshold
max_gap = thr_75_log * 0.5
gaps_log = np.arange(0, max_gap, max_gap / 25)
coverages, f1s = [], []
for gap in gaps_log:
    dec = np.where(upper_log < (thr_75_log - gap), 'Low',
          np.where(lower_log > (thr_75_log + gap), 'High', 'Uncertain'))
    mask = dec != 'Uncertain'
    cov = mask.mean()
    if mask.sum() > 0:
        pb = (dec == 'High').astype(int)
        f1 = f1_score(true_label[mask], pb[mask], average='binary', zero_division=0)
    else:
        f1 = 0.0
    coverages.append(cov)
    f1s.append(f1)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.plot(gaps_log, [c * 100 for c in coverages], 'b-o', ms=3, label='Coverage %')
ax2.plot(gaps_log, f1s, 'r-s', ms=3, label='F1 (definite)')
ax1.set_xlabel('Gap around threshold (log-citation units)')
ax1.set_ylabel('Coverage (%)', color='b')
ax2.set_ylabel('F1 on definite predictions', color='r')
ax1.set_title(f'Coverage vs F1 tradeoff  [{int(Q_LOW*100)}/50/{int(Q_HIGH*100)} % CI]')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.show()
print('Gap=0: raw q25–q75 interval used. Larger gap = stricter definite predictions.')

## 9. Interval width by ground-truth label

In [ ]:
width_log = upper_log - lower_log
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Distribution of interval width (log-space) by true label
for lbl, mask, color in [
    ('Low Impact (true)', true_label == 0, 'steelblue'),
    ('High Impact (true)', true_label == 1, 'tomato'),
]:
    axes[0].hist(width_log[mask], bins=40, alpha=0.6, label=lbl, color=color)
axes[0].axvline(np.median(width_log), color='k', ls='--', lw=1, label='Median width')
axes[0].set_xlabel('Interval width (log-citation units)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Prediction interval width by true label  [50 % CI: q{int(Q_LOW*100)}/q{int(Q_HIGH*100)}]')
axes[0].legend()

# Scatter: q50 log-prediction vs truth
cap = np.quantile(y_test_log, 0.99)
colors = np.where(decisions == 'High', 'tomato',
         np.where(decisions == 'Low', 'steelblue', 'gold'))
axes[1].scatter(
    np.clip(y_test_log, 0, cap),
    np.clip(median_log, 0, cap),
    c=colors, alpha=0.3, s=10
)
axes[1].axhline(thr_75_log, color='k', lw=1, ls='--', label=f'log1p threshold = {thr_75_log:.2f}')
axes[1].axvline(thr_75_log, color='k', lw=1, ls='--')
axes[1].plot([0, cap], [0, cap], 'k:', lw=0.8, alpha=0.4)
axes[1].set_xlim(0, cap * 1.05); axes[1].set_ylim(0, cap * 1.05)
axes[1].set_xlabel('Actual log1p(citations)')
axes[1].set_ylabel('q50 log prediction')
axes[1].set_title('Prediction vs actual (colour = decision)')
from matplotlib.patches import Patch
axes[1].legend(handles=[
    Patch(color='tomato',    label='High (definite)'),
    Patch(color='steelblue', label='Low (definite)'),
    Patch(color='gold',      label='Uncertain'),
] + axes[1].get_legend_handles_labels()[0], loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## 10. Save models

In [ ]:
QUANTILE_DIR = MODELS_DIR / 'quantile'
QUANTILE_DIR.mkdir(parents=True, exist_ok=True)

for q, m in quantile_models.items():
    path = QUANTILE_DIR / f'lgbm_q{int(q*100):02d}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(m, f)
    print(f'Saved {path.name}')

# Save threshold (same as nb32 artifact, confirming consistency)
with open(MODELS_DIR / 'citation_threshold_p75.pkl', 'wb') as f:
    pickle.dump(thr_75, f)
print(f'Threshold saved: {thr_75:.0f} citations (p75 of training set)')

# Save the q50 model as the primary regression model (drop-in for nb32)
with open(MODELS_DIR / 'lightgbm_q50.pkl', 'wb') as f:
    pickle.dump(quantile_models[0.50], f)
print('q50 model saved as lightgbm_q50.pkl (drop-in for existing lightgbm.pkl)')

## 11. Regression metrics for q50

In [ ]:
pred_log_q50 = quantile_models[0.50].predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test_log, pred_log_q50))
mae  = mean_absolute_error(y_test_log, pred_log_q50)
r2   = r2_score(y_test_log, pred_log_q50)
rho  = spearmanr(y_test_log, pred_log_q50).statistic

print('q50 regression metrics (log-space):')
print(f'  RMSE    : {rmse:.4f}')
print(f'  MAE     : {mae:.4f}')
print(f'  R²      : {r2:.4f}')
print(f'  Spearman: {rho:.4f}')

## Summary

| Artifact | Path |
|---|---|
| q25 model | `models/regression/quantile/lgbm_q25.pkl` |
| q50 model | `models/regression/quantile/lgbm_q50.pkl` |
| q75 model | `models/regression/quantile/lgbm_q75.pkl` |
| q50 drop-in | `models/regression/lightgbm_q50.pkl` |
| p75 threshold | `models/regression/citation_threshold_p75.pkl` |

**Key design choices:**
* **50 % CI (q25/q75)** instead of 80 % (q10/q90): the 80 % interval is too wide
  relative to the threshold in log-space, causing zero High predictions.
* **Train 2010–2017, Test 2018–2020**: avoids distribution shift from using
  citation-immature papers (2020–2025) as the test set.
* Threshold set at p75 of training citations (~43), evaluated in log-space.
* Coverage vs F1 can be tuned by adding a gap around the threshold (Section 8).